# 3.3 线性代数：把推荐数据放进有形状的盒子

> 3.0 基础课程：先直觉，再符号，再数字代入，再用代码和图形核对。

## Goal

理解标量、向量、矩阵、张量及其轴和形状，亲手计算逐元素乘法、点积、矩阵乘法、范数、余弦、embedding、低秩分解与注意力形状。

## Setup 与数据边界

本课不加载用户行为数据。代码中的小数组都是带标签的 **数学教学对象**，只用于验证公式，绝不冒充交互、曝光、标签或行为序列。

In [ ]:
from pathlib import Path
import os, sys
import numpy as np
import matplotlib.pyplot as plt
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ARTIFACT_ROOT = Path(os.environ.get("RECSYS_ARTIFACT_ROOT", PROJECT_ROOT)).expanduser().resolve()
sys.path.insert(0, str(PROJECT_ROOT))
TEACHING_OBJECTS_ONLY = True
print({"project_root": str(PROJECT_ROOT), "artifact_root": str(ARTIFACT_ROOT), "kind": "curriculum", "teaching_objects_only": True})

from matplotlib import font_manager

# Matplotlib 默认的 DejaVu Sans 不包含中文字形。优先选择容器中的 Noto CJK
# （镜像已安装 fonts-noto-cjk），其次是宿主机常见中文字体；从字体根源解决，
# 而不是用 warnings.filterwarnings 掩盖 missing glyph 警告。
cjk_candidates = ('Noto Sans CJK SC', 'Noto Sans CJK JP', 'PingFang SC', 'Hiragino Sans GB',
                  'Microsoft YaHei', 'SimHei', 'Arial Unicode MS', 'Songti SC')
installed_fonts = {font.name for font in font_manager.fontManager.ttflist}
cjk_font = next((name for name in cjk_candidates if name in installed_fonts), None)
if cjk_font:
    plt.rcParams['font.sans-serif'] = [cjk_font, 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
print('图表字体:', cjk_font or '未找到中文字体（请安装 fonts-noto-cjk）')

## 学习路径

先辨认形状，再辨认运算；先手算一个格子，再让 NumPy 批量算；最后把矩阵乘法连接到 embedding、低秩模型和 Q/K/V 注意力。

## 符号表

| 符号 | 含义 | 形状示例 |
|---|---|---|
| $a$ | 标量，一个数 | `()` |
| $x$ | 向量，一排数 | $(d,)$ |
| $X$ | 矩阵，行×列 | $(B,d)$ |
| $T$ | 张量，三轴及以上 | $(B,L,d)$ |
| $X^\top$ | 转置，交换行列 | $(d,B)$ |
| $\|x\|_2$ | 向量长度 | 标量 |

> 本课所有数组都是**数学教学对象**，不是用户、曝光或交互记录。

<a id="tensors-shapes"></a>
## 1. 标量、向量、矩阵、张量、轴与形状

可把形状想成盒子的尺寸。$X\in\mathbb R^{3\times2}$ 有 3 行 2 列；序列张量 $(B,L,d)$ 依次表示批大小、序列长度、每个位置的表示维数。轴不是抽象术语：沿轴 1 求和，就是把每行的列加起来。

### 数字代入 1：沿轴求和

$$X=\begin{bmatrix}1&2\\3&4\\5&6\end{bmatrix}.$$

沿列方向合并每行得到 $[1+2,3+4,5+6]=[3,7,11]$；沿行方向合并每列得到 $[1+3+5,2+4+6]=[9,12]$。

<a id="elementwise-dot"></a>
## 2. 逐元素运算、点积、范数与余弦

逐元素乘法保留形状：$[1,2]\odot[3,4]=[3,8]$。点积再求和：$[1,2]\cdot[3,4]=1\times3+2\times4=11$。长度 $\|x\|_2=\sqrt{\sum x_j^2}$；余弦除去长度，只比较方向。

### 数字代入 2：长度与余弦

$x=[3,4]$ 的长度为 $5$；$y=[6,8]$ 长度为 $10$，点积 $50$，所以余弦 $50/(5\times10)=1$：长度不同但方向完全相同。

In [ ]:
# Demo 1：形状、轴、逐元素、点积与余弦。
X = np.array([[1., 2.], [3., 4.], [5., 6.]])
x, y = np.array([3., 4.]), np.array([6., 8.])
cosine = (x @ y) / (np.linalg.norm(x) * np.linalg.norm(y))
print({"shape": X.shape, "sum_axis_1": X.sum(axis=1).tolist(),
       "elementwise": (x*y).tolist(), "dot": float(x@y), "cosine": cosine})
fig, ax = plt.subplots(figsize=(5, 4))
ax.quiver([0,0], [0,0], [x[0],y[0]], [x[1],y[1]], angles="xy", scale_units="xy", scale=1)
ax.set(xlim=(0,9), ylim=(0,9), title="同方向、不同长度的教学向量"); ax.grid(); plt.show()

<a id="matmul-embedding"></a>
## 3. 矩阵乘法与 embedding 查表

若 $A$ 形状 $(m,n)$、$B$ 形状 $(n,p)$，中间的 $n$ 必须相同，结果是 $(m,p)$。结果第 $(r,c)$ 格是 $A$ 第 $r$ 行与 $B$ 第 $c$ 列的点积。one-hot 乘 embedding 表等价于查一行。

### 数字代入 3：手算矩阵的一个格子

$$A=\begin{bmatrix}1&2\\0&1\end{bmatrix},\quad B=\begin{bmatrix}3&4\\5&6\end{bmatrix}.$$

左上角为 $1\times3+2\times5=13$，右上角为 $1\times4+2\times6=16$，所以 $AB=\begin{bmatrix}13&16\\5&6\end{bmatrix}$。one-hot $[0,1,0]$ 乘三行 embedding 表，会精确取出第二行。

加权和也是矩阵乘法：权重 $[0.2,0.8]$ 加权两个向量 $[1,0]$、$[0,1]$ 得 $[0.2,0.8]$。归一化常把权重总和变为 1，方便解释“分票”。

<a id="low-rank-attention"></a>
## 4. 低秩、转置与 Q/K/V 形状

低秩近似 $R\approx PQ^\top$ 用两个小矩阵描述大矩阵：$P$ 每行是一个用户坐标，$Q$ 每行是一个物品坐标。注意力中 $QK^\top$ 把每个 query 与每个 key 做点积；若 $Q,K,V$ 均为 $(B,L,d)$，则分数为 $(B,L,L)$，乘 $V$ 后回到 $(B,L,d)$。

### 数字代入 4：低秩重构一个格子

用户坐标 $p=[1,2]$，物品坐标 $q=[3,-1]$，重构分数为 $p^\top q=1\times3+2\times(-1)=1$。这只是数学对象，不是声称某用户真的给了 1 分。

In [ ]:
# Demo 2：embedding 查表、低秩重构和注意力形状。
embedding_table = np.array([[1.,0.], [0.,2.], [-1.,1.]])
one_hot = np.array([0.,1.,0.])
print("one-hot @ table =", one_hot @ embedding_table)
R_math = np.array([[5.,4.,1.], [4.,3.,1.], [1.,1.,5.]])  # 数学教学矩阵，不是交互行
U, s, Vt = np.linalg.svd(R_math, full_matrices=False)
rank2 = (U[:, :2] * s[:2]) @ Vt[:2]
Q = np.arange(2*3*4, dtype=float).reshape(2,3,4) / 10
K, V = Q + .1, Q - .1
scores = Q @ K.transpose(0,2,1)
output = scores @ V
print({"rank2_shape": rank2.shape, "attention_scores": scores.shape, "output": output.shape})
fig, axes = plt.subplots(1,2,figsize=(8,3))
axes[0].imshow(R_math); axes[0].set_title("教学矩阵")
axes[1].imshow(rank2); axes[1].set_title("rank-2 近似")
plt.tight_layout(); plt.show()

## 常见误区

- `*` 与 `@` 相同：前者通常逐元素，后者是矩阵乘法。
- 形状能广播就一定语义正确：程序能跑不代表轴对齐正确。
- embedding 维度有固定含义：各坐标通常联合学习，不能单独命名。
- 低秩重构是恢复事实：它是模型估计，会有误差和偏差。

## 算法回链

- [ItemCF：矩阵共现与余弦](/notebooks/4_2_collaborative_filtering)
- [BiasMF：低秩分解](/notebooks/4_3_matrix_factorization)
- [DSSM：批量 user-item 点积](/notebooks/5_2_dssm)
- [MIND：多个兴趣的加权聚合](/notebooks/5_3_mind)
- [SASRec：Q/K/V 与因果注意力](/notebooks/5_4_sasrec)

## Checks

1. $(8,20,16)$ 的序列张量中三个轴各可能代表什么？
2. 为什么 $(3,4)@(4,5)$ 得 $(3,5)$？
3. 余弦为 1 是否表示两个向量完全相等？

In [ ]:
assert X.shape == (3,2)
assert np.isclose(cosine, 1.0)
assert (np.array([[1,2],[0,1]]) @ np.array([[3,4],[5,6]])).tolist() == [[13,16],[5,6]]
assert scores.shape == (2,3,3) and output.shape == (2,3,4)
print("PASS：轴、点积、矩阵乘法、低秩和注意力形状检查通过。")

## Next Steps

下一课把模型看作函数的复合：线性代数负责算出预测，微积分负责回答参数该往哪边改。